In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

sns.set_style("whitegrid")
pd.set_option("display.max_columns",50)
pd.set_option("display.float_format","{:.2f}".format)

print("Libraries loaded.")

Libraries loaded.


In [7]:
df_raw=pd.read_csv(r"C:\Users\rampr\Downloads\bikes.csv")

In [10]:
df_raw["owner"].unique()

<StringArray>
['first owner', 'third owner', 'second owner', 'fourth owner or more']
Length: 4, dtype: str

In [13]:
df_raw.dtypes

model_name      str
model_year    int64
kms_driven      str
owner           str
location        str
mileage         str
power           str
price         int64
dtype: object

In [14]:
df_raw.duplicated().sum()

np.int64(0)

In [15]:
df_raw.isnull().sum()

model_name     0
model_year     0
kms_driven     0
owner          0
location      19
mileage       11
power         31
price          0
dtype: int64

In [16]:
df_raw['price'].describe()

count      7857.00
mean     106791.34
std      138926.12
min           0.00
25%       42000.00
50%       75000.00
75%      125000.00
max     3000000.00
Name: price, dtype: float64

In [17]:
df_raw.columns

Index(['model_name', 'model_year', 'kms_driven', 'owner', 'location',
       'mileage', 'power', 'price'],
      dtype='str')

In [19]:
df_raw[['kms_driven','mileage','power']].columns

Index(['kms_driven', 'mileage', 'power'], dtype='str')

In [36]:
def get_number(value):
    if pd.isna(value):
        return np.nan
    value=str(value).replace(',','')
    number='' #string
    dot_used=False

    for i in value:
        if i.isdigit():
            number+=i
        elif i=='.' and not dot_used:
            dot_used=True
        elif number:
            break
    return float(number) if number else np.nan


In [31]:
df_raw.columns

Index(['model_name', 'model_year', 'kms_driven', 'owner', 'location',
       'mileage', 'power', 'price'],
      dtype='str')

In [34]:
column_name=['mileage', 'power', 'price', 'model_year', 'kms_driven']
for i in column_name:
    df_raw[i]= df_raw[i].apply(get_number)

In [35]:
df_raw

,model_name,model_year,kms_driven,owner,location,mileage,power,price
0,Bajaj Avenger Cruise 220 2017,2017.00,17000.00,first owner,hyderabad,35.00,19.00,63500.00
1,Royal Enfield Classic 350cc 2016,2016.00,50000.00,first owner,hyderabad,35.00,1980.00,115000.00
2,Hyosung GT250R 2012,2012.00,14795.00,first owner,hyderabad,30.00,28.00,300000.00
3,Bajaj Dominar 400 ABS 2017,2017.00,28.00,first owner,pondicherry,28.00,3450.00,100000.00
4,Jawa Perak 330cc 2020,2020.00,2000.00,first owner,bangalore,NaN,30.00,197500.00
...,...,...,...,...,...,...,...,...
7852,Yamaha YZF-R15 150cc 2011,2011.00,7000.00,first owner,agra,42.00,16.00,55000.00
7853,Bajaj Discover 100cc 2015,2015.00,80.00,first owner,delhi,80.00,77.00,28000.00
7854,Bajaj Pulsar 180cc 2016,2016.00,6407.00,first owner,bangalore,65.00,17.00,61740.00
7855,Bajaj V15 150cc 2016,2016.00,7524.00,first owner,bangalore,57.00,1180.00,49000.00


In [37]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 7857 entries, 0 to 7856
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   model_name  7857 non-null   str    
 1   model_year  7857 non-null   float64
 2   kms_driven  7854 non-null   float64
 3   owner       7857 non-null   str    
 4   location    7838 non-null   str    
 5   mileage     7083 non-null   float64
 6   power       7826 non-null   float64
 7   price       7857 non-null   float64
dtypes: float64(5), str(3)
memory usage: 491.2 KB


In [38]:
df_raw.duplicated().sum()

np.int64(0)

In [46]:
df=df_raw.dropna(subset='price')

In [47]:
df=df[df['price']>0]

In [52]:
df[df['model_year']<1990]

,model_name,model_year,kms_driven,owner,location,mileage,power,price
1691,Royal Enfield Bullet 350 cc 1982,1982.00,22000.00,first owner,pune,NaN,NaN,150000.00
1716,Royal Enfield Bullet 350 cc 1982,1982.00,16245.00,first owner,delhi,NaN,NaN,70000.00
2211,Bajaj Pulsar NS 200cc 2013,1950.00,35.00,first owner,noida,35.00,2319.00,55000.00
2487,Royal Enfield Bullet 350 cc 1982,1982.00,3000.00,second owner,lalitpur,NaN,NaN,130000.00
2638,Ideal Jawa Yezdi Delux 250 cc 1978,1978.00,85000.00,first owner,chikamaglur,NaN,NaN,100000.00
2714,Royal Enfield Bullet 350 cc 1982,1982.00,21000.00,first owner,nashik,NaN,NaN,170000.00
3088,Royal Enfield Bullet 350 cc 1982,1982.00,10000.00,first owner,mumbai,NaN,NaN,60000.00
3429,Royal Enfield‎ Bullet 350cc 1970,1970.00,5000.00,third owner,pune,25.00,1980.00,190000.00
4174,Royal Enfield Standard 350cc 1985,1985.00,40.00,third owner,kolhapur,40.00,18.00,125000.00
4258,Bajaj Pulsar NS 200cc 2013,1950.00,35.00,third owner,khalilabad,35.00,2319.00,55000.00


In [54]:
pd.Timestamp.today().year

2026

In [56]:
if 'model_year' in df.columns:
    current_year= pd.Timestamp.today().year
    df['model_year']= df['model_year'].between(1998, current_year)